### persistence Setup
Run this cell to mount your Google Drive. Change the `DRIVE_PATH` if you want to save to a specific folder. This is the most reliable way to prevent data loss during long runs.

In [3]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Define a persistent directory on your Drive
DRIVE_PATH = '/content/drive/MyDrive/Quantum_Experiments'
os.makedirs(DRIVE_PATH, exist_ok=True)

print(f"Persistent storage initialized at: {DRIVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent storage initialized at: /content/drive/MyDrive/Quantum_Experiments


In [ ]:
# 1. Install and Import PennyLane first to prevent circular import issues
!pip install --upgrade pennylane
import pennylane as qml
from pennylane import numpy as np

import time
import pickle
import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
from google.colab import files

# ==========================================
# CELL 1: Imports & Setup
# ==========================================
warnings.filterwarnings('ignore')

# Experiment Constants
SYSTEMS = [8, 10, 12]
SUPPORTS = {
    8: [2, 3, 5, 6, 8],
    10: [2, 4, 6, 8, 10],
    12: [2, 5, 7, 10, 12]
}
MAX_DEPTH = 25
N_SAMPLES = 15
THRESHOLD = 1e-2
NOISE_LEVELS = [0.000, 0.005, 0.020, 0.050]

# ==========================================
# CELL 2: Architectures
# ==========================================
def compute_tau_bp_threshold(var_curve, depth_range, threshold):
    for var, d in zip(var_curve, depth_range):
        if var < threshold:
            return d
    return None

def BrickWall_HEA_noise(n_qubits, params, depth, p_noise):
    idx = 0
    for d in range(depth):
        for q in range(n_qubits):
            qml.RY(params[idx], wires=q)
            idx += 1
        if d % 2 == 0:
            for q in range(0, n_qubits - 1, 2):
                qml.CNOT(wires=[q, q + 1])
                if p_noise > 0:
                    qml.DepolarizingChannel(p_noise, wires=q)
                    qml.DepolarizingChannel(p_noise, wires=q + 1)
        else:
            for q in range(1, n_qubits - 1, 2):
                qml.CNOT(wires=[q, q + 1])
                if p_noise > 0:
                    qml.DepolarizingChannel(p_noise, wires=q)
                    qml.DepolarizingChannel(p_noise, wires=q + 1)

def LongRange_HEA_noise(n_qubits, params, depth, p_noise):
    idx = 0
    for d in range(depth):
        for q in range(n_qubits):
            qml.RY(params[idx], wires=q)
            idx += 1
        half_n = n_qubits // 2
        if d % 2 == 0:
            for q in range(half_n):
                target = (q + half_n) % n_qubits
                qml.CNOT(wires=[q, target])
                if p_noise > 0:
                    qml.DepolarizingChannel(p_noise, wires=q)
                    qml.DepolarizingChannel(p_noise, wires=target)
        else:
            for q in range(half_n):
                control = (q + 1) % n_qubits
                target = (control + half_n) % n_qubits
                qml.CNOT(wires=[control, target])
                if p_noise > 0:
                    qml.DepolarizingChannel(p_noise, wires=control)
                    qml.DepolarizingChannel(p_noise, wires=target)

def make_observable(k):
    op = qml.PauliZ(0)
    for q in range(1, k):
        op = op @ qml.PauliZ(q)
    return op

# ==========================================
# CELL 3: Run Experiments (Explicit Persistence)
# ==========================================
CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DRIVE_TARGET = "/content/drive/MyDrive/Quantum_Experiments"

PKL_NAME = "noise_checkpoint.pkl"
CSV_NAME = "noise_partial.csv"

POSSIBLE_LOAD_PATHS = [
    os.path.join(DRIVE_TARGET, PKL_NAME),
    os.path.join(CHECKPOINT_DIR, PKL_NAME),
    "/content/noise_checkpoint.pkl"
]

LOAD_PATH = None
for path in POSSIBLE_LOAD_PATHS:
    if os.path.exists(path):
        LOAD_PATH = path
        print(f"SUCCESS: Found checkpoint at {path}. Resuming...")
        break

if LOAD_PATH:
    with open(LOAD_PATH, "rb") as f:
        checkpoint = pickle.load(f)
    rows = checkpoint.get("rows", [])
    variance_curves = checkpoint.get("variance_curves", {})
    completed = checkpoint.get("completed", set())
    print(f"Resumed: {len(completed)} configurations already finished.")
else:
    print("No existing progress found. Starting fresh.")
    rows, variance_curves, completed = [], {}, set()

arch_map = {"Brick-Wall": BrickWall_HEA_noise, "Long-Range": LongRange_HEA_noise}
TOTAL_CONFIGS = len(arch_map) * len(NOISE_LEVELS) * sum(len(v) for v in SUPPORTS.values())

for arch_name, ansatz_fn in arch_map.items():
    for p in NOISE_LEVELS:
        for n in SYSTEMS:
            dev = qml.device("default.mixed", wires=n)
            for k in SUPPORTS[n]:
                config_key = (arch_name, p, n, k)
                if config_key in completed:
                    continue

                print(f"\nProgress: {len(completed)}/{TOTAL_CONFIGS}")
                print(f"Running: {arch_name} | p={p} | n={n} | k={k}")

                @qml.qnode(dev, diff_method="backprop")
                def cost_fn(params, L):
                    ansatz_fn(n, params, L, p)
                    return qml.expval(make_observable(k))

                var_curve = []
                for depth in range(1, MAX_DEPTH + 1):
                    grads_all = []
                    for _ in range(N_SAMPLES):
                        params = np.random.uniform(0, 2*np.pi, size=depth * n, requires_grad=True)
                        grad = qml.grad(cost_fn)(params, depth)
                        grads_all.extend(np.array(grad[0] if isinstance(grad, tuple) else grad).flatten())
                    variance = np.var(grads_all)
                    var_curve.append(variance)

                tau = compute_tau_bp_threshold(var_curve, range(1, MAX_DEPTH + 1), THRESHOLD) or (MAX_DEPTH + 1)
                variance_curves[config_key] = var_curve
                rows.append({"architecture": arch_name, "noise": p, "n": n, "k": k, "tau": tau})
                completed.add(config_key)

                local_pkl = os.path.join(CHECKPOINT_DIR, PKL_NAME)
                with open(local_pkl, "wb") as f:
                    pickle.dump({"rows": rows, "variance_curves": variance_curves, "completed": completed}, f)

                try:
                    shutil.copy2(local_pkl, os.path.join(DRIVE_TARGET, PKL_NAME))
                    pd.DataFrame(rows).to_csv(os.path.join(DRIVE_TARGET, CSV_NAME), index=False)
                    print(f"Progress synced to Drive.")
                except Exception as e:
                    print(f"Drive sync failed: {e}")

noise_df = pd.DataFrame(rows)
print("\nAll experiments completed successfully.")

SUCCESS: Found checkpoint at /content/drive/MyDrive/Quantum_Experiments/noise_checkpoint.pkl. Resuming...
Resumed: 6 configurations already finished.

Progress: 6/120
Running: Brick-Wall | p=0.0 | n=10 | k=4


### Auto-Download & Notification
Run the cell below after your experiment loop. It will zip the results and trigger a browser download automatically when the simulation finishes.

In [ ]:
from google.colab import files
import shutil

# 1. Final Save to persistent Drive
final_results_path = os.path.join(DRIVE_PATH, "final_results")
if os.path.exists('./results'):
    shutil.copytree('./results', final_results_path, dirs_exist_ok=True)

# 2. Create a zip of everything
shutil.make_archive('experiment_results_backup', 'zip', './checkpoints')

# 3. Trigger automatic browser download
print("Attempting automatic download of backup zip...")
files.download('experiment_results_backup.zip')

print("Check your Google Drive folder for the permanent copies!")

In [ ]:
# ==========================================
# CELL 4: Fit Empirical Scaling Laws..,..
# ==========================================
print("\nFitting Empirical Law: tau = A * (nk)^(-c) for all models...")

def calculate_metrics(y_true, y_pred, model, n_obs, k_params):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)

    # Manually calculate AICc for small sample sizes
    aic = model.aic
    aicc = aic + (2 * k_params * (k_params + 1)) / (n_obs - k_params - 1)

    return {
        "MAE": mae, "RMSE": rmse, "MAPE (%)": mape,
        "R2": r2, "AIC": aic, "AICc": aicc, "BIC": model.bic
    }

model_summaries = []
fitted_models = {}

for name, group in noise_df.groupby(['architecture', 'noise']):
    Y = np.log(group['tau'])
    nk_product = group['n'] * group['k']
    X = sm.add_constant(np.log(nk_product))

    model = sm.OLS(Y, X).fit()
    fitted_models[name] = model

    log_A = model.params.iloc[0]
    c = -model.params.iloc[1]
    A = np.exp(log_A)

    # Predictions
    pred_tau = A * (nk_product ** (-c))
    noise_df.loc[group.index, 'predicted_tau'] = pred_tau
    noise_df.loc[group.index, 'residual'] = group['tau'] - pred_tau

    metrics = calculate_metrics(group['tau'], pred_tau, model, n_obs=len(Y), k_params=2)

    summary_dict = {
        "architecture": name[0],
        "noise": name[1],
        "A": A,
        "c": c
    }
    summary_dict.update(metrics)
    model_summaries.append(summary_dict)

noise_model_summary = pd.DataFrame(model_summaries)
print("\nScaling Laws Fitted. Summary:")
display(noise_model_summary[['architecture', 'noise', 'A', 'c', 'R2', 'AICc']])


# ==========================================
# CELL 5: Validation - Leave-One-Out Cross-Validation (LOOCV)
# ==========================================
print("\nRunning Leave-One-Out Cross-Validation (LOOCV)...")

loocv_results = []

for name, group in noise_df.groupby(['architecture', 'noise']):
    group = group.reset_index(drop=True)
    loo_preds = []

    for i in range(len(group)):
        # Split train/test
        test_row = group.iloc[i]
        train_df = group.drop(i)

        # Fit on N-1 points
        Y_train = np.log(train_df['tau'])
        X_train = sm.add_constant(np.log(train_df['n'] * train_df['k']))
        loo_model = sm.OLS(Y_train, X_train).fit()

        # Recover parameters
        log_A_loo = loo_model.params.iloc[0]
        c_loo = -loo_model.params.iloc[1]
        A_loo = np.exp(log_A_loo)

        # Predict on the 1 left out point
        test_nk = test_row['n'] * test_row['k']
        pred = A_loo * (test_nk ** (-c_loo))
        loo_preds.append(pred)

    # Calculate LOOCV metrics for this group
    true_vals = group['tau'].values
    loo_mae = mean_absolute_error(true_vals, loo_preds)
    loo_rmse = np.sqrt(mean_squared_error(true_vals, loo_preds))
    loo_mape = np.mean(np.abs((true_vals - loo_preds) / true_vals)) * 100

    loocv_results.append({
        "architecture": name[0],
        "noise": name[1],
        "LOO_MAE": loo_mae,
        "LOO_RMSE": loo_rmse,
        "LOO_MAPE (%)": loo_mape
    })

loocv_df = pd.DataFrame(loocv_results)
noise_model_summary = pd.merge(noise_model_summary, loocv_df, on=['architecture', 'noise'])
print("\nLOOCV Complete.")


# ==========================================
# CELL 6: Statistical Analysis (Wilcoxon Signed-Rank Test)
# ==========================================
print("\n" + "="*60)
print("STATISTICAL TESTING: NOISE=0 VS NOISE>0")
print("="*60)

for arch in arch_map.keys():
    print(f"\nArchitecture: {arch}")
    base_tau = noise_df[(noise_df['architecture'] == arch) & (noise_df['noise'] == 0.0)]['tau'].values

    for p in [n for n in NOISE_LEVELS if n > 0]:
        noisy_tau = noise_df[(noise_df['architecture'] == arch) & (noise_df['noise'] == p)]['tau'].values

        # Ensure we are paired correctly
        w_stat, p_val = stats.wilcoxon(base_tau, noisy_tau)
        mean_diff = np.mean(noisy_tau - base_tau)

        significance = "Significant" if p_val < 0.05 else "Not Significant"
        print(f"  Noise {p:.3f} | Mean Diff: {mean_diff:>6.2f} layers | p-value: {p_val:.4e} | {significance}")


# ==========================================
# CELL 7: Figures
# ==========================================
print("\nGenerating Figures...")

# Standardize colors
colors = {0.000: 'black', 0.005: 'blue', 0.020: 'orange', 0.050: 'red'}

# --- Figure 1: Variance Curves ---
fig1, axes1 = plt.subplots(1, 2, figsize=(14, 5))
example_n = 10
example_k = 4
depth_range = range(1, MAX_DEPTH + 1)

for idx, arch in enumerate(arch_map.keys()):
    for p in NOISE_LEVELS:
        # Retrieve the variance curve using the exact config_key tuple
        var_data = variance_curves[(arch, p, example_n, example_k)]
        # Pad the curve if early stopping triggered
        if len(var_data) < MAX_DEPTH:
            var_data.extend([var_data[-1]] * (MAX_DEPTH - len(var_data)))
        axes1[idx].plot(depth_range, var_data, label=f'p={p}', color=colors[p])

    axes1[idx].axhline(THRESHOLD, color='gray', linestyle='--', label='Threshold')
    axes1[idx].set_yscale('log')
    axes1[idx].set_title(f'Fig 1: {arch} Variance Decay (n={example_n}, k={example_k})')
    axes1[idx].set_xlabel('Depth (L)')
    axes1[idx].set_ylabel('Gradient Variance')
    axes1[idx].legend()
    axes1[idx].grid(True, alpha=0.3)

# --- Figure 2: Tau versus k ---
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
for idx, arch in enumerate(arch_map.keys()):
    for p in NOISE_LEVELS:
        subset = noise_df[(noise_df['architecture'] == arch) & (noise_df['noise'] == p) & (noise_df['n'] == 10)]
        axes2[idx].plot(subset['k'], subset['tau'], marker='o', label=f'p={p}', color=colors[p])

    axes2[idx].set_title(f'Fig 2: {arch} Tau vs k (n=10)')
    axes2[idx].set_xlabel('Observable Weight (k)')
    axes2[idx].set_ylabel('Tau_BP')
    axes2[idx].legend()
    axes2[idx].grid(True, alpha=0.3)

# --- Figure 3 & 4: Exponent c and Avg Tau versus Noise ---
fig34, axes34 = plt.subplots(1, 2, figsize=(14, 5))

for arch in arch_map.keys():
    subset = noise_model_summary[noise_model_summary['architecture'] == arch]
    axes34[0].plot(subset['noise'], subset['c'], marker='o', label=arch)

    # Calculate avg tau per noise level
    avg_taus = []
    for p in NOISE_LEVELS:
        avg_t = noise_df[(noise_df['architecture'] == arch) & (noise_df['noise'] == p)]['tau'].mean()
        avg_taus.append(avg_t)
    axes34[1].plot(NOISE_LEVELS, avg_taus, marker='o', label=arch)

axes34[0].set_title('Fig 3: Scaling Exponent (c) vs Noise')
axes34[0].set_xlabel('Depolarizing Probability (p)')
axes34[0].set_ylabel('Exponent c')
axes34[0].legend()
axes34[0].grid(True, alpha=0.3)

axes34[1].set_title('Fig 4: Average Tau vs Noise')
axes34[1].set_xlabel('Depolarizing Probability (p)')
axes34[1].set_ylabel('Average Tau_BP (Over all n, k)')
axes34[1].legend()
axes34[1].grid(True, alpha=0.3)

# --- Figure 5: Observed vs Predicted ---
fig5, axes5 = plt.subplots(1, 2, figsize=(14, 5))
for idx, arch in enumerate(arch_map.keys()):
    subset = noise_df[noise_df['architecture'] == arch]
    axes5[idx].scatter(subset['tau'], subset['predicted_tau'], c=subset['noise'], cmap='viridis', alpha=0.7)

    min_val = min(subset['tau'].min(), subset['predicted_tau'].min())
    max_val = max(subset['tau'].max(), subset['predicted_tau'].max())
    axes5[idx].plot([min_val, max_val], [min_val, max_val], 'k--', label='Perfect Fit')

    axes5[idx].set_title(f'Fig 5: {arch} Observed vs Predicted Tau')
    axes5[idx].set_xlabel('Observed Tau_BP')
    axes5[idx].set_ylabel('Predicted Tau_BP')
    axes5[idx].grid(True, alpha=0.3)
    axes5[idx].legend()

# --- Figure 6: Residuals ---
fig6, axes6 = plt.subplots(1, 2, figsize=(14, 5))
for idx, arch in enumerate(arch_map.keys()):
    subset = noise_df[noise_df['architecture'] == arch]
    scatter = axes6[idx].scatter(subset['tau'], subset['residual'], c=subset['noise'], cmap='viridis', alpha=0.7)
    axes6[idx].axhline(0, color='black', linestyle='--')

    axes6[idx].set_title(f'Fig 6: {arch} Residual Plot')
    axes6[idx].set_xlabel('Observed Tau_BP')
    axes6[idx].set_ylabel('Residual')
    axes6[idx].grid(True, alpha=0.3)
    fig6.colorbar(scatter, ax=axes6[idx], label='Noise (p)')

plt.tight_layout()
plt.show()


# ==========================================
# CELL 8: Save to CSV
# ==========================================
output_dir = "./results"
os.makedirs(output_dir, exist_ok=True)

df_path = os.path.join(output_dir, "noise_df.csv")
noise_df.to_csv(df_path, index=False)

summary_path = os.path.join(output_dir, "noise_model_summary.csv")
noise_model_summary.to_csv(summary_path, index=False)

print(f"\nResults successfully saved to:")
print(f"- {df_path}")

print(f"- {summary_path}")